# Joint APTOS + EyePACS training of the P grader ("J") — record §93

**One change from P: the training data.** ConvNeXt-Tiny from ImageNet, CORN head, the cached direct-resize 512 frames,
P's augmentation, weighted CORN, AdamW 1e-4, fixed 0.5 decode — all unchanged. No crop, no new augmentation.

- **Epoch:** all 2,921 APTOS training images + 5,842 EyePACS training images (a rotating slice of one seeded shuffle;
  a pure function of seed and epoch, so a resumed run sees the same epochs). Batch 16.
- **Class weights:** P's formula on the grade counts of that mixture (0.6456 / 1.8421 / 1.1954 / 2.8293 / 2.5958).
- **Checkpoint selection:** the highest mean of the APTOS validation QWK (730 images) and the EyePACS validation QWK
  (3,512 images), both full sets, scored every epoch.
- **Seeds, in sequence in ONE runtime:** 42 first (cell 5), then 123 and 2026 one after the other (cell 6). Do not train two
  seeds at the same time from different runtimes. Seeds 123 and 2026 run only if seed 42 is at most 0.01 below P on APTOS validation and at most
  0.02 below the EyePACS-only model on EyePACS validation; the trainer refuses otherwise.

Not read: the EyePACS held-out test, IDRiD, DDR.

**Limitations that travel with the result:** batch 16 here against batch 2 for P (the comparison with P confounds joint
data with batch size); both validation sets select the checkpoint; EyePACS validation is in-domain for this model, so
nothing here is evidence of generalisation to unseen data.

Disk: the EyePACS frame cache (51.4 GiB) and the APTOS RGB frames (10.7 GiB) are copied to `/content` — about 62 GiB of
the T4's 71 GB. Time on a T4: about 30 minutes of staging, then about 6.5 minutes per epoch; early stopping is expected
after 2.5 – 3.5 hours (5.5 hours at the 50-epoch cap). The run is resumable: after a disconnect, run the cells again.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / Stage-4 weights needed; no external test set is used.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_train as at
import e1_data as ed, e1_train as e1t, e2_control as e2
import eyepacs_adaptation_data as ea, eyepacs_adaptation_train as eat, ep_aptos_train as ep, joint_train as jt
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] SWITCHES -- everything expensive is OFF. Set a flag to True (and type its phrase) to allow that step. ====
STAGE_DATA   = False   # copy the EyePACS frame cache (51.4 GiB) and the APTOS RGB frames (10.7 GiB) to /content
RUN_SEED_42  = False   # train (or resume) seed 42, evaluate BEST, compare, apply the continuation rule
RUN_SEEDS_2  = False   # seeds 123 and 2026 -- refused by the trainer unless seed 42 met the continuation rule
CONFIRM = {"seed_42": "", "seeds_2": ""}
PHRASES = {"seed_42": "TRAIN THE JOINT MODEL SEED 42", "seeds_2": "TRAIN JOINT SEEDS 123 AND 2026"}

def confirmed(step, flag):
    if flag and CONFIRM[step] != PHRASES[step]:
        raise RuntimeError(f"{step}: the flag is True but CONFIRM['{step}'] is not the phrase {PHRASES[step]!r}")
    return bool(flag)
print({k: v for k, v in globals().items() if k.startswith(("RUN_", "STAGE_"))})

In [ ]:
# ==== [3] INPUTS AND PROTOCOL (reads manifests and small files only; trains nothing) ====
EXP = colab_config.DRIVE.experiments_root
ROWS = ea.read_split(posixpath.join(REPO_DIR, "dataset_splits"), ea.MANIFEST_SHA256)      # refuses any other manifest
SPLIT = ea.split_report(ROWS)
assert SPLIT["images"] == {"train": 31610, "val": 3512, "total": 35122} and SPLIT["patients_in_both_splits"] == 0
assert jt.PROTOCOL == dict(at.P_PROTOCOL, batch_size=16) == eat.PROTOCOL and jt.SEEDS == (42, 123, 2026)
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256
DRIVE_CACHE = posixpath.join(colab_config.DATASET_ROOT, "EyePACS", "adaptation_frames_v1")
LOCAL_CACHE = "/content/eyepacs_adaptation_frames_v1"
EXPECTED_MODEL_SHA = "cb5fc7a8d370af7d2ae191cadaebdde858f71820d147fb76f852b757766f8ad8"   # names the APTOS bundle only; Stage 4 is not used
GEN4 = cache.stage4_generation_id(EXPECTED_MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2_e1", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
# the two reference tables of the comparison: P (same seed) on APTOS validation, the EyePACS-only model on EyePACS validation
P_TABLE = posixpath.join(EXP, "PL_ConvNeXtPriors", "2026-09-28_05-22-44", "P_convnext_rgb", "seed_{seed}", "evaluation", "per_sample_best.csv")
EYEPACS_ONLY_TABLE = posixpath.join(eat.run_dir_for(EXP, ea.MANIFEST_SHA256), "metrics", "per_sample_best.csv")
assert all(os.path.exists(P_TABLE.format(seed=s)) for s in jt.SEEDS) and os.path.exists(EYEPACS_ONLY_TABLE)
print("protocol", jt.PROTOCOL)
print("selection:", jt.SELECTION)
print("class weights (recorded)", jt.APPROVED_CLASS_WEIGHTS, "| epoch: 2,921 APTOS + 5,842 EyePACS | order tag", jt.ORDER_TAG)
print("runs:", {s: jt.state(jt.run_dir_for(EXP, s)) for s in jt.SEEDS}, "| commit", REPO_COMMIT)

In [ ]:
# ==== [4] DATA: Drive -> local disk (every file checked by SHA-256), then the four parts and the weight check ====
PARTS = FRAMES = None
if STAGE_DATA:
    print(ea.stage_cache(DRIVE_CACHE, LOCAL_CACHE))
    ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=EXPECTED_MODEL_SHA)
    ed.stage_inputs(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
if os.path.exists(posixpath.join(LOCAL_CACHE, "index.json")) and os.path.isdir(LOCAL_ROOTS["bundle_v2"]):
    EYEPACS_FRAMES = ea.FrameCache(LOCAL_CACHE, ROWS, check_hashes=False, log=print)      # staging hashed every shard
    assert EYEPACS_FRAMES.report["frames"] == {"train": 31610, "val": 3512}
    BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                            expected_stage4_sha256=EXPECTED_MODEL_SHA, roots=LOCAL_ROOTS)
    _tr, _va, _split_sha = msr.verify_split()                                            # the authoritative APTOS split
    _empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
    assert _split_sha == v2cfg.SPLIT_SHA256 == BUNDLE.bundle["split_sha256"]
    assert list(BUNDLE.train_ids) == [str(i) for i, _ in _tr if str(i) not in _empty] and len(BUNDLE.train_ids) == 2921
    assert list(BUNDLE.val_ids) == [str(i) for i, _ in _va if str(i) not in _empty] and len(BUNDLE.val_ids) == 730
    APTOS_TRAIN, APTOS_VAL = ep.entries(BUNDLE)
    PARTS = (APTOS_TRAIN, APTOS_VAL, EYEPACS_FRAMES.entries("train"), EYEPACS_FRAMES.entries("val"))
    jt.check_data(*PARTS)                                                                # no validation image in a training part
    FRAMES = jt.JointFrames(ep.BundleFrames(BUNDLE), EYEPACS_FRAMES, [i for i, _ in APTOS_TRAIN + APTOS_VAL],
                            [i for i, _ in PARTS[2] + PARTS[3]])
    IDENTITIES = {"aptos": {"split_sha256": BUNDLE.bundle["split_sha256"], "bundle_id": BUNDLE.bundle["bundle_id"], "fingerprint": BUNDLE.fingerprint},
                  "eyepacs": {"split_manifest_sha256": EYEPACS_FRAMES.manifest_sha256, "fingerprint": EYEPACS_FRAMES.fingerprint}}
    for _s in jt.SEEDS:                                                                  # recorded weights against the mixture actually sampled
        _expected, _weights, _sampled = jt.class_weights(PARTS[0], PARTS[2], _s)
        print(f"seed {_s}: weights {[round(w, 4) for w in _weights]} | sampled-mixture weights {[round(w, 4) for w in _sampled['weights']]}"
              f" | largest difference {_sampled['max_abs_difference']:.4f}")
    print("APTOS train / val", len(PARTS[0]), len(PARTS[1]), "| EyePACS train / val", len(PARTS[2]), len(PARTS[3]))
else:
    print("data not staged (set STAGE_DATA = True once per runtime)")

In [ ]:
# ==== [5] SEED 42 -- THIS TRAINS (about 2.5 - 3.5 h on a T4; resumable). Then BEST is evaluated and compared. ====
def imagenet_arrays():
    tf.keras.backend.clear_session()
    return pl.load_reference(PRETRAINED)[1]

def run_joint(seed):
    tf.keras.backend.clear_session()
    _, _arrays = pl.load_reference(PRETRAINED)
    return jt.run_seed(seed, EXP, FRAMES, PARTS, _arrays, IDENTITIES,
                       {"p_aptos": P_TABLE.format(seed=seed), "eyepacs_only": EYEPACS_ONLY_TABLE},
                       repo_dir=REPO_DIR, staging_root="/content/joint_staging")

def show(result):
    b, c = result["best"], result["comparisons"]
    print(f"seed {result['seed']}: BEST epoch index {b['checkpoint']['epoch_index']} of {result['epochs_scored']} scored"
          f" | selection metric {b['checkpoint']['selection_metric']:.4f} | stop: {result['stop']}")
    for _name, _key, _ref in (("aptos_val", "aptos_val_vs_p", "P (same seed)"), ("eyepacs_val", "eyepacs_val_vs_eyepacs_only", "EyePACS-only model")):
        _m, _d = b["sets"][_name], c[_key]
        print(f"  {_name}: QWK {_m['qwk']:.4f} | {_ref} {_d['reference_qwk']:.4f} | difference {_d['difference']:+.4f} [{_d['ci_95'][0]:+.4f}, {_d['ci_95'][1]:+.4f}]")
        print(f"     AUROC cuts { {k: round(v, 3) for k, v in _m['auroc_cuts'].items()} } | recall { [round(_m['recall_per_grade'][str(g)] if str(g) in _m['recall_per_grade'] else _m['recall_per_grade'][g], 3) for g in range(5)] }")
        print(f"     referable sens / spec {_m['referable_sensitivity']:.3f} / {_m['referable_specificity']:.3f} | ECE {_m['ece_top_label']:.3f}")
    if "continuation" in result:
        print("  continuation rule:", json.dumps(result["continuation"]))
    for _l in result["limitations"]:
        print("  limitation:", _l)

if confirmed("seed_42", RUN_SEED_42):
    assert FRAMES is not None, "stage the data first (cell 4)"
    RESULT_42 = run_joint(42)
    show(RESULT_42)
else:
    print("seed 42 not run (RUN_SEED_42 = False):", jt.state(jt.run_dir_for(EXP, 42)))

In [ ]:
# ==== [6] SEEDS 123 THEN 2026, in sequence -- only after seed 42 met the continuation rule (the trainer refuses otherwise) ====
if confirmed("seeds_2", RUN_SEEDS_2):
    assert FRAMES is not None, "stage the data first (cell 4)"
    # seed 42 is returned as recorded, then 123 and 2026 are trained one after the other in this runtime
    RESULTS = jt.run_arm("direct", EXP, FRAMES, PARTS, imagenet_arrays, IDENTITIES,
                         lambda seed: {"p_aptos": P_TABLE.format(seed=seed), "eyepacs_only": EYEPACS_ONLY_TABLE},
                         repo_dir=REPO_DIR, staging_root="/content/joint_staging", on_result=show)
else:
    print("seeds 123 / 2026 not run (RUN_SEEDS_2 = False):", {s: jt.state(jt.run_dir_for(EXP, s)) for s in (123, 2026)})

In [ ]:
# ==== [7] PER-EPOCH VALIDATION HISTORY of a seed (reads what the run wrote) ====
_dir = posixpath.join(jt.run_dir_for(EXP, 42), "dual_validation")
if os.path.isdir(_dir):
    for _f in sorted(os.listdir(_dir)):
        _e = json.load(open(posixpath.join(_dir, _f)))
        print(f"epoch {_e['epoch']:>2}: APTOS {_e['val_QWK_aptos']:.4f} | EyePACS {_e['val_QWK_eyepacs']:.4f} | mean {_e['selection_metric']:.4f}")
else:
    print("no epoch has been scored yet")